In [ ]:
!pip install torch torchvision tqdm scikit-learn matplotlib -q


In [ ]:
import os
import random
import shutil
from tqdm import tqdm

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader as TorchDataLoader
from torchvision import datasets, transforms

from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt


In [ ]:
import os
import zipfile

# اسم الملف اللي انت رفعته (اتأكد ان اسمه كدة بالظبط)
zip_path = '/content/drive/MyDrive/cv project/archive (1).zip'

# المكان اللي هنفك فيه الداتا
extract_path = '/content/drive/MyDrive/cv project/add'

if os.path.exists(zip_path):
    print("جاري فك الضغط... ⏳")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('/content/') # هيفك في الروت عشان الفولدر اسمه اصلا processed_data
    print("تم فك الضغط بنجاح! ✅")
else:
    print("❌ ملف الـ zip مش موجود، اتأكد انك رفعته في القائمة الجانبية")

جاري فك الضغط... ⏳
تم فك الضغط بنجاح! ✅


In [ ]:
class Config:
    def __init__(self):
        # عدّل ده حسب مكان الداتا في كولاب
        self.data_root = "/content/drive/MyDrive/cv project/processed_data - Copy"  # جوهها angry, happy, ...

        self.train_dir = os.path.join(self.data_root, "train")
        self.val_dir   = os.path.join(self.data_root, "val")

        self.num_classes   = 7
        self.batch_size    = 64
        self.num_epochs    = 20
        self.learning_rate = 1e-3
        self.weight_decay  = 1e-4
        self.num_workers   = 2
        self.pin_memory    = True
        self.device        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.seed          = 42

        self.save_path     = "/content/emotion_model_best.pt"
        self.print_every   = 1

        self.emotion_labels = {
            0: "angry",
            1: "disgust",
            2: "fear",
            3: "happy",
            4: "sad",
            5: "surprise",
            6: "neutral",
        }

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


In [ ]:
class DataSplitter:
    def __init__(self, dataset_root, train_ratio=0.8, seed=42):
        self.dataset_root = dataset_root
        self.train_ratio  = train_ratio
        self.seed         = seed

        random.seed(self.seed)

        self.train_dir = os.path.join(self.dataset_root, "train")
        self.val_dir   = os.path.join(self.dataset_root, "val")

    def _list_classes(self):
        classes = []
        for name in os.listdir(self.dataset_root):
            path = os.path.join(self.dataset_root, name)
            if os.path.isdir(path) and name not in ["train", "val"]:
                classes.append(name)
        return classes

    def _create_dirs(self, classes):
        for cls in classes:
            os.makedirs(os.path.join(self.train_dir, cls), exist_ok=True)
            os.makedirs(os.path.join(self.val_dir, cls), exist_ok=True)

    def split(self):
        classes = self._list_classes()
        print("Classes:", classes)
        self._create_dirs(classes)

        for cls in classes:
            class_path = os.path.join(self.dataset_root, cls)
            images = [
                f for f in os.listdir(class_path)
                if os.path.isfile(os.path.join(class_path, f))
            ]

            print(f"\nClass {cls} - {len(images)} images")
            random.shuffle(images)

            train_count = int(len(images) * self.train_ratio)
            train_imgs = images[:train_count]
            val_imgs   = images[train_count:]

            for img in tqdm(train_imgs, desc=f"train {cls}", ncols=60):
                src = os.path.join(class_path, img)
                dst = os.path.join(self.train_dir, cls, img)
                shutil.copy2(src, dst)

            for img in tqdm(val_imgs, desc=f"val   {cls}", ncols=60):
                src = os.path.join(class_path, img)
                dst = os.path.join(self.val_dir, cls, img)
                shutil.copy2(src, dst)

        print("\n>>> Done splitting dataset!")


In [ ]:
class EmotionDataModule:
    def __init__(self, cfg: Config):
        self.cfg = cfg

    def _build_transforms(self):
        train_tf = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.Resize((48, 48)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,)),
        ])

        val_tf = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.Resize((48, 48)),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,)),
        ])

        return train_tf, val_tf

    def setup(self):
        train_tf, val_tf = self._build_transforms()

        self.train_dataset = datasets.ImageFolder(
            root=self.cfg.train_dir,
            transform=train_tf
        )
        self.val_dataset = datasets.ImageFolder(
            root=self.cfg.val_dir,
            transform=val_tf
        )

        print("Train classes:", self.train_dataset.classes)
        print("Val   classes:", self.val_dataset.classes)
        print("Train samples:", len(self.train_dataset))
        print("Val samples  :", len(self.val_dataset))

    def train_dataloader(self):
        return TorchDataLoader(
            self.train_dataset,
            batch_size=self.cfg.batch_size,
            shuffle=True,
            num_workers=self.cfg.num_workers,
            pin_memory=self.cfg.pin_memory,
        )

    def val_dataloader(self):
        return TorchDataLoader(
            self.val_dataset,
            batch_size=self.cfg.batch_size,
            shuffle=False,
            num_workers=self.cfg.num_workers,
            pin_memory=self.cfg.pin_memory,
        )


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2),
        )

    def forward(self, x):
        return self.block(x)


class EmotionCNN(nn.Module):
    def __init__(self, num_classes: int = 7):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(1, 32),    # 48 -> 24
            ConvBlock(32, 64),   # 24 -> 12
            ConvBlock(64, 128),  # 12 -> 6
            ConvBlock(128, 256), # 6 -> 3
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = self.classifier(x)
        return x


In [ ]:
class Trainer:
    def __init__(self, model, train_loader, val_loader, cfg: Config):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.cfg = cfg

        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.Adam(
            self.model.parameters(),
            lr=self.cfg.learning_rate,
            weight_decay=self.cfg.weight_decay,
        )
        # التعديل هنا: شيلنا verbose=True
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode="min", factor=0.5, patience=3
        )
        self.best_val_acc = 0.0

    def train_one_epoch(self, epoch: int):
        self.model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in self.train_loader:
            images = images.to(self.cfg.device)
            labels = labels.to(self.cfg.device)

            self.optimizer.zero_grad()
            outputs = self.model(images)
            loss = self.criterion(outputs, labels)
            loss.backward()
            self.optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        return running_loss / total, correct / total

    def evaluate(self):
        self.model.eval()
        running_loss, correct, total = 0.0, 0, 0

        with torch.no_grad():
            for images, labels in self.val_loader:
                images = images.to(self.cfg.device)
                labels = labels.to(self.cfg.device)

                outputs = self.model(images)
                loss = self.criterion(outputs, labels)

                running_loss += loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        return running_loss / total, correct / total

    def fit(self):
        print("Training on:", self.cfg.device)

        for epoch in range(1, self.cfg.num_epochs + 1):
            train_loss, train_acc = self.train_one_epoch(epoch)
            val_loss, val_acc = self.evaluate()

            # التعديل هنا: الـ scheduler مش محتاج verbose لأنه بيشتغل صامت دلوقتي
            self.scheduler.step(val_loss)

            if epoch % self.cfg.print_every == 0:
                print(
                    f"Epoch [{epoch}/{self.cfg.num_epochs}] "
                    f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} "
                    f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
                )
                # لو عايز تطبع الـ Learning Rate يدوي عشان تتابع التغيير:
                current_lr = self.optimizer.param_groups[0]['lr']
                print(f"Current LR: {current_lr}")

            if val_acc > self.best_val_acc:
                self.best_val_acc = val_acc
                torch.save(self.model.state_dict(), self.cfg.save_path)
                print(f">>> New best model saved! Val Acc = {self.best_val_acc:.4f}")

        print("Training finished. Best Val Acc:", self.best_val_acc)

In [ ]:
def main():
    cfg = Config()
    set_seed(cfg.seed)

    # لو أول مرة ولسه مفيش train/val -> اعمل split
    if not os.path.exists(cfg.train_dir) or not os.listdir(cfg.train_dir):
        print("No train/val folders found. Splitting dataset...")
        splitter = DataSplitter(cfg.data_root, train_ratio=0.8, seed=cfg.seed)
        splitter.split()
    else:
        print("Train/Val folders already exist, skipping split.")

    data = EmotionDataModule(cfg)
    data.setup()
    train_loader = data.train_dataloader()
    val_loader   = data.val_dataloader()

    model = EmotionCNN(num_classes=cfg.num_classes).to(cfg.device)
    print(model)

    trainer = Trainer(model, train_loader, val_loader, cfg)
    trainer.fit()

    # load best model and eval confusion matrix
    model.load_state_dict(torch.load(cfg.save_path, map_location=cfg.device))
    class_names = data.train_dataset.classes  # ['angry', 'disgust', ...]
    evaluate_confusion_matrix(model, val_loader, class_names, cfg.device)


main()


No train/val folders found. Splitting dataset...
Classes: ['fear', 'disgust', 'angry', 'neutral', 'happy', 'sad', 'surprise']

Class fear - 5928 images


val   fear: 100%|███████| 1186/1186 [00:14<00:00, 82.94it/s]



Class disgust - 5920 images


val   disgust: 100%|████| 1184/1184 [00:14<00:00, 82.62it/s]



Class angry - 5920 images


...

val   sad: 100%|████████| 1303/1303 [00:15<00:00, 82.69it/s]



Class surprise - 5920 images


val   surprise: 100%|███| 1184/1184 [00:14<00:00, 81.60it/s]



>>> Done splitting dataset!
Train classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
Val   classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
Train samples: 39825
Val samples  : 9957
EmotionCNN(
  (features): Sequential(
    (0): ConvBlock(
      (block): Sequential(
        (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (5): ReLU(inplace=True)
        (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      )
    )
    (1): ConvBlock(
      (block): Sequential(
        (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(64, eps=1e-05, mome

KeyboardInterrupt: 

In [ ]:
import os
import shutil
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader as TorchDataLoader
from torchvision import transforms, datasets, models
from tqdm import tqdm

# ==========================================
# 1. Configuration
# ==========================================
class Config:
    def __init__(self):
        self.data_root = "/content/processed_data"  # تأكد إن المسار صح
        self.train_dir = os.path.join(self.data_root, "train")
        self.val_dir   = os.path.join(self.data_root, "val")

        self.num_classes   = 7
        self.batch_size    = 32    # قللناه شوية عشان الصور كبرت (224)
        self.num_epochs    = 15    # ResNet بيتعلم بسرعة
        self.learning_rate = 1e-4  # Learning rate أهدى شوية للـ Fine-tuning
        self.weight_decay  = 1e-4
        self.num_workers   = 2
        self.device        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.save_path     = "/content/resnet18_emotion.pt"
        self.print_every   = 1

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# ==========================================
# 2. Data Loader (ResNet Specific)
# ==========================================
class EmotionDataModule:
    def __init__(self, cfg: Config):
        self.cfg = cfg

    def _build_transforms(self):
        # ResNet محتاج صور 224x224 و 3 قنوات ألوان
        train_tf = transforms.Compose([
            transforms.Grayscale(num_output_channels=3), # تكرار القناة الرمادي 3 مرات
            transforms.Resize((224, 224)),               # تكبير الصورة
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.ColorJitter(brightness=0.2, contrast=0.2), # إضافة شوية "نويز" للإضاءة
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # تطبيع ImageNet
        ])

        val_tf = transforms.Compose([
            transforms.Grayscale(num_output_channels=3),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        return train_tf, val_tf

    def setup(self):
        train_tf, val_tf = self._build_transforms()
        self.train_dataset = datasets.ImageFolder(root=self.cfg.train_dir, transform=train_tf)
        self.val_dataset = datasets.ImageFolder(root=self.cfg.val_dir, transform=val_tf)

        print(f"Train samples: {len(self.train_dataset)}")
        print(f"Val samples:   {len(self.val_dataset)}")

    def get_loaders(self):
        train_loader = TorchDataLoader(self.train_dataset, batch_size=self.cfg.batch_size, shuffle=True, num_workers=self.cfg.num_workers)
        val_loader = TorchDataLoader(self.val_dataset, batch_size=self.cfg.batch_size, shuffle=False, num_workers=self.cfg.num_workers)
        return train_loader, val_loader

# ==========================================
# 3. Model: Pre-trained ResNet18
# ==========================================
def get_resnet_model(num_classes=7, device='cpu'):
    print("Loading Pre-trained ResNet18...")
    # تحميل الموديل بالأوزان الجاهزة
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

    # تجميد الطبقات الأولى (اختياري، بس خلينا ندرب كله عشان الدقة تكون أحسن للوشوش)
    # for param in model.parameters():
    #     param.requires_grad = False

    # تغيير آخر طبقة عشان تطلع 7 مشاعر بس
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)

    return model.to(device)

# ==========================================
# 4. Trainer
# ==========================================
class Trainer:
    def __init__(self, model, train_loader, val_loader, cfg: Config):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.cfg = cfg

        self.criterion = nn.CrossEntropyLoss()
        # AdamW usually works better for transfer learning
        self.optimizer = optim.AdamW(self.model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(self.optimizer, mode="min", factor=0.5, patience=2)
        self.best_val_acc = 0.0

    def train_epoch(self):
        self.model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in tqdm(self.train_loader, desc="Training", leave=False):
            images, labels = images.to(self.cfg.device), labels.to(self.cfg.device)

            self.optimizer.zero_grad()
            outputs = self.model(images)
            loss = self.criterion(outputs, labels)
            loss.backward()
            self.optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        return running_loss / total, correct / total

    def evaluate(self):
        self.model.eval()
        running_loss, correct, total = 0.0, 0, 0

        with torch.no_grad():
            for images, labels in self.val_loader:
                images, labels = images.to(self.cfg.device), labels.to(self.cfg.device)
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)

                running_loss += loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        return running_loss / total, correct / total

    def fit(self):
        print(f"Starting training on {self.cfg.device}...")
        for epoch in range(1, self.cfg.num_epochs + 1):
            train_loss, train_acc = self.train_epoch()
            val_loss, val_acc = self.evaluate()

            self.scheduler.step(val_loss)

            print(f"Epoch [{epoch}/{self.cfg.num_epochs}] "
                  f"Train Loss: {train_loss:.4f} | Acc: {train_acc:.1%} | "
                  f"Val Loss: {val_loss:.4f} | Acc: {val_acc:.1%}")

            if val_acc > self.best_val_acc:
                self.best_val_acc = val_acc
                torch.save(self.model.state_dict(), self.cfg.save_path)
                print(f">>> New Best Model Saved: {val_acc:.1%}")

# ==========================================
# 5. Main Execution
# ==========================================
if __name__ == "__main__":
    cfg = Config()
    set_seed()

    # Load Data
    data_module = EmotionDataModule(cfg)
    data_module.setup()
    train_loader, val_loader = data_module.get_loaders()

    # Load Model
    model = get_resnet_model(num_classes=cfg.num_classes, device=cfg.device)

    # Train
    trainer = Trainer(model, train_loader, val_loader, cfg)
    trainer.fit()

FileNotFoundError: [Errno 2] No such file or directory: '/content/processed_data/train'

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms, datasets, models
from torch.utils.data import DataLoader as TorchDataLoader

# ==========================================
# 1. Updated Data Module (Essential for Pre-trained Models)
# ==========================================
class EmotionDataModule:
    def __init__(self, cfg):
        self.cfg = cfg

    def _build_transforms(self):
        # EfficientNet محتاج صور 224x224 و 3 قنوات ألوان
        # وتطبيع (Normalization) بأرقام ImageNet
        train_tf = transforms.Compose([
            transforms.Grayscale(num_output_channels=3), # مهم جداً: 3 قنوات
            transforms.Resize((224, 224)),               # مهم جداً: تكبير الصورة
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),               # زودنا الدوران شوية
            transforms.ColorJitter(brightness=0.2, contrast=0.2), # لعب في الإضاءة
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        val_tf = transforms.Compose([
            transforms.Grayscale(num_output_channels=3),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        return train_tf, val_tf

    def setup(self):
        train_tf, val_tf = self._build_transforms()

        self.train_dataset = datasets.ImageFolder(
            root=self.cfg.train_dir,
            transform=train_tf
        )
        self.val_dataset = datasets.ImageFolder(
            root=self.cfg.val_dir,
            transform=val_tf
        )

        print("Train classes:", self.train_dataset.classes)
        print("Val   classes:", self.val_dataset.classes)
        print("Train samples:", len(self.train_dataset))
        print("Val samples  :", len(self.val_dataset))

    def train_dataloader(self):
        return TorchDataLoader(
            self.train_dataset,
            batch_size=self.cfg.batch_size,
            shuffle=True,
            num_workers=self.cfg.num_workers,
            pin_memory=self.cfg.pin_memory,
        )

    def val_dataloader(self):
        return TorchDataLoader(
            self.val_dataset,
            batch_size=self.cfg.batch_size,
            shuffle=False,
            num_workers=self.cfg.num_workers,
            pin_memory=self.cfg.pin_memory,
        )

# ==========================================
# 2. EfficientNet Model
# ==========================================
class EmotionEfficientNet(nn.Module):
    def __init__(self, num_classes=7, pretrained=True):
        super(EmotionEfficientNet, self).__init__()

        # تحميل EfficientNet-B0 (خفيف وقوي)
        # لو عايز أقوى ممكن تجرب efficientnet_b2 بس أتقل
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        self.base_model = models.efficientnet_b0(weights=weights)

        # تجميد الطبقات الأولى (اختياري)
        # جرب تدرب من غير تجميد الأول (Unfrozen) عشان ياخد على صور الوشوش
        # for param in self.base_model.features.parameters():
        #     param.requires_grad = False

        # تعديل الطبقة الأخيرة (Classifier)
        # EfficientNet الـ classifier بتاعه عبارة عن Sequential
        # آخر طبقة فيه (رقم 1) هي اللي بتطلع النتايج، هنغيرها لعدد الكلاسات بتاعنا (7)
        in_features = self.base_model.classifier[1].in_features

        self.base_model.classifier[1] = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.base_model(x)

# ==========================================
# طريقة الاستخدام في كود التدريب
# ==========================================
# model = EmotionEfficientNet(num_classes=7).to(device)

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader as TorchDataLoader
from torchvision import transforms, datasets, models
from tqdm import tqdm

# ==========================================
# 1. Configuration
# ==========================================
class Config:
    def __init__(self):
        # Change this path to your data location
        self.data_root = "/content/drive/MyDrive/cv project/processed_data - Copy"
        self.train_dir = os.path.join(self.data_root, "train")
        self.val_dir   = os.path.join(self.data_root, "val")

        self.num_classes   = 7
        self.batch_size    = 32    # 32 is safe for 224x224 images on Colab
        self.num_epochs    = 20    # EfficientNet needs time to fine-tune
        self.learning_rate = 3e-4  # Good starting point for pre-trained models
        self.weight_decay  = 1e-4
        self.num_workers   = 2
        self.device        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.save_path     = "/content/efficientnet_emotion.pt"

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# ==========================================
# 2. Data Module (Updated for EfficientNet)
# ==========================================
class EmotionDataModule:
    def __init__(self, cfg: Config):
        self.cfg = cfg

    def _build_transforms(self):
        # EfficientNet requires 224x224 and 3 Channels (RGB)
        train_tf = transforms.Compose([
            transforms.Grayscale(num_output_channels=3),
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        val_tf = transforms.Compose([
            transforms.Grayscale(num_output_channels=3),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        return train_tf, val_tf

    def setup(self):
        train_tf, val_tf = self._build_transforms()
        self.train_dataset = datasets.ImageFolder(root=self.cfg.train_dir, transform=train_tf)
        self.val_dataset = datasets.ImageFolder(root=self.cfg.val_dir, transform=val_tf)

        print(f"Train samples: {len(self.train_dataset)}")
        print(f"Val samples:   {len(self.val_dataset)}")

    def get_loaders(self):
        train_loader = TorchDataLoader(self.train_dataset, batch_size=self.cfg.batch_size, shuffle=True, num_workers=self.cfg.num_workers)
        val_loader = TorchDataLoader(self.val_dataset, batch_size=self.cfg.batch_size, shuffle=False, num_workers=self.cfg.num_workers)
        return train_loader, val_loader

# ==========================================
# 3. Model: EfficientNet-B0
# ==========================================
class EmotionEfficientNet(nn.Module):
    def __init__(self, num_classes=7, pretrained=True):
        super(EmotionEfficientNet, self).__init__()

        print("Loading EfficientNet-B0...")
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        self.base_model = models.efficientnet_b0(weights=weights)

        # Modify the classifier head
        # The classifier in EfficientNet is a Sequential block, index 1 is the Linear layer
        in_features = self.base_model.classifier[1].in_features

        self.base_model.classifier[1] = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.base_model(x)

# ==========================================
# 4. Trainer
# ==========================================
class Trainer:
    def __init__(self, model, train_loader, val_loader, cfg: Config):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.cfg = cfg

        self.criterion = nn.CrossEntropyLoss()

        # AdamW is generally better for Transfer Learning than standard Adam
        self.optimizer = optim.AdamW(
            self.model.parameters(),
            lr=cfg.learning_rate,
            weight_decay=cfg.weight_decay
        )

        # Scheduler to reduce LR when validation loss plateaus
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode="min", factor=0.5, patience=2
        )

        self.best_val_acc = 0.0

    def train_epoch(self):
        self.model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        loop = tqdm(self.train_loader, desc="Training", leave=False)

        for images, labels in loop:
            images = images.to(self.cfg.device)
            labels = labels.to(self.cfg.device)

            # Forward pass
            self.optimizer.zero_grad()
            outputs = self.model(images)
            loss = self.criterion(outputs, labels)

            # Backward pass
            loss.backward()
            self.optimizer.step()

            # Metrics
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            # Update progress bar
            loop.set_postfix(loss=loss.item())

        return running_loss / total, correct / total

    def evaluate(self):
        self.model.eval()
        running_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in self.val_loader:
                images = images.to(self.cfg.device)
                labels = labels.to(self.cfg.device)

                outputs = self.model(images)
                loss = self.criterion(outputs, labels)

                running_loss += loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        return running_loss / total, correct / total

    def fit(self):
        print(f"🚀 Starting training on {self.cfg.device}...")

        for epoch in range(1, self.cfg.num_epochs + 1):
            train_loss, train_acc = self.train_epoch()
            val_loss, val_acc = self.evaluate()

            # Step the scheduler
            self.scheduler.step(val_loss)

            print(f"Epoch [{epoch}/{self.cfg.num_epochs}] "
                  f"Train Loss: {train_loss:.4f} | Acc: {train_acc:.2%} | "
                  f"Val Loss: {val_loss:.4f} | Acc: {val_acc:.2%}")

            # Save best model
            if val_acc > self.best_val_acc:
                self.best_val_acc = val_acc
                torch.save(self.model.state_dict(), self.cfg.save_path)
                print(f"🔥 New Best Model Saved! ({val_acc:.2%}) -> {self.cfg.save_path}")

        print("\n✅ Training Complete.")
        print(f"🏆 Best Validation Accuracy: {self.best_val_acc:.2%}")

# ==========================================
# 5. Main Execution
# ==========================================
if __name__ == "__main__":
    # 1. Setup Config & Seed
    cfg = Config()
    set_seed()

    # 2. Setup Data
    if not os.path.exists(cfg.train_dir):
        print("❌ Error: Data not found. Please upload and unzip data first.")
    else:
        data_module = EmotionDataModule(cfg)
        data_module.setup()
        train_loader, val_loader = data_module.get_loaders()

        # 3. Setup Model
        model = EmotionEfficientNet(num_classes=cfg.num_classes).to(cfg.device)

        # 4. Train
        trainer = Trainer(model, train_loader, val_loader, cfg)
        trainer.fit()

Train samples: 39825
Val samples:   9957
Loading EfficientNet-B0...
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 94.3MB/s]


🚀 Starting training on cuda...


Epoch [1/20] Train Loss: 1.0722 | Acc: 60.41% | Val Loss: 0.8236 | Acc: 68.74%
🔥 New Best Model Saved! (68.74%) -> /content/efficientnet_emotion.pt


Epoch [2/20] Train Loss: 0.8234 | Acc: 69.47% | Val Loss: 0.7404 | Acc: 72.82%
🔥 New Best Model Saved! (72.82%) -> /content/efficientnet_emotion.pt


Epoch [3/20] Train Loss: 0.7255 | Acc: 73.16% | Val Loss: 0.7055 | Acc: 74.41%
🔥 New Best Model Saved! (74.41%) -> /content/efficientnet_emotion.pt


Epoch [4/20] Train Loss: 0.6532 | Acc: 75.66% | Val Loss: 0.6714 | Acc: 75.90%
🔥 New Best Model Saved! (75.90%) -> /content/efficientnet_emotion.pt


Epoch [5/20] Train Loss: 0.5919 | Acc: 78.00% | Val Loss: 0.6556 | Acc: 76.48%
🔥 New Best Model Saved! (76.48%) -> /content/efficientnet_emotion.pt


Training:  94%|█████████▍| 1174/1245 [04:49<00:17,  4.17it/s, loss=0.401]

# DeepFace

In [ ]:
!wget https://github.com/derronqi/yolo-face/releases/download/v0.0.0/yolov8n-face.pt
